# SVM

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, recall_score, precision_score, roc_auc_score


# load data
df = pd.read_csv("weather_prediction_dataset.csv")


# Filter for BASEL 
basel_cols = [c for c in df.columns if c.startswith("BASEL_")]
feature_cols = [c for c in basel_cols if c != "BASEL_cloud_cover"]

X = df[feature_cols]


# Creating binary target for use of classification metrics in validation 
target_raw = df["BASEL_cloud_cover"]
df["target"] = (target_raw > 4).astype(int)
y = df["target"]

#trail/val/test/split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

#train svm
svm = SVC(kernel="rbf", probability=True, random_state=42)
svm.fit(X_train, y_train)

#predictions
y_pred_train = svm.predict(X_train)
y_pred_val = svm.predict(X_val)
y_pred_test = svm.predict(X_test)

# Probabilities for AUC
y_prob_test = svm.predict_proba(X_test)[:, 1]


# metrics
metrics = {
    "train_accuracy": accuracy_score(y_train, y_pred_train),
    "validation_accuracy": accuracy_score(y_val, y_pred_val),
    "test_accuracy": accuracy_score(y_test, y_pred_test),
    "f1": f1_score(y_test, y_pred_test),
    "kappa": cohen_kappa_score(y_test, y_pred_test),
    "recall": recall_score(y_test, y_pred_test),
    "precision": precision_score(y_test, y_pred_test),
    "auc": roc_auc_score(y_test, y_prob_test),
}

print("SVM Results:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

SVM Results:
train_accuracy: 0.8842
validation_accuracy: 0.8777
test_accuracy: 0.8925
f1: 0.9225
kappa: 0.7474
recall: 0.9213
precision: 0.9237
auc: 0.9567


# KNN

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, recall_score, precision_score, roc_auc_score

# load data
df = pd.read_csv("weather_prediction_dataset.csv")

# filter for BASEL
basel_cols = [c for c in df.columns if c.startswith("BASEL_")]
feature_cols = [c for c in basel_cols if c != "BASEL_cloud_cover"]

X = df[feature_cols]

# Creating binary target for use of classification metrics in validation
target_raw = df["BASEL_cloud_cover"]
df["target"] = (target_raw > 4).astype(int)
y = df["target"]

# trail/val/test/split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# train knn
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train, y_train)

# predictions
y_pred_train = knn.predict(X_train)
y_pred_val = knn.predict(X_val)
y_pred_test = knn.predict(X_test)

# Probabilities for AUC
y_prob_test = knn.predict_proba(X_test)[:, 1]


# metrics

metrics = {
    "train_accuracy": accuracy_score(y_train, y_pred_train),
    "validation_accuracy": accuracy_score(y_val, y_pred_val),
    "test_accuracy": accuracy_score(y_test, y_pred_test),
    "f1": f1_score(y_test, y_pred_test),
    "kappa": cohen_kappa_score(y_test, y_pred_test),
    "recall": recall_score(y_test, y_pred_test),
    "precision": precision_score(y_test, y_pred_test),
    "auc": roc_auc_score(y_test, y_prob_test),
}

print("KNN Results:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


KNN Results:
train_accuracy: 0.9014
validation_accuracy: 0.8777
test_accuracy: 0.8780
f1: 0.9115
kappa: 0.7150
recall: 0.9055
precision: 0.9176
auc: 0.9445


# Bagging

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    recall_score, precision_score, roc_auc_score
)

# load data
df = pd.read_csv("weather_prediction_dataset.csv")

# filter for BASEL
basel_cols = [c for c in df.columns if c.startswith("BASEL_")]
feature_cols = [c for c in basel_cols if c != "BASEL_cloud_cover"]

X = df[feature_cols]

# Creating binary target for classification metrics
target_raw = df["BASEL_cloud_cover"]
df["target"] = (target_raw > 4).astype(int)
y = df["target"]

# train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# Define the Bagging model
bag_model = BaggingClassifier(
    estimator=DecisionTreeClassifier(),   # updated parameter name
    n_estimators=50,
    random_state=42,
    n_jobs=-1,
    bootstrap=True
)

# train
bag_model.fit(X_train, y_train)

# predictions
y_pred_train = bag_model.predict(X_train)
y_pred_val = bag_model.predict(X_val)
y_pred_test = bag_model.predict(X_test)

# Probabilities for AUC
y_prob_test = bag_model.predict_proba(X_test)[:, 1]

# metrics
metrics = {
    "train_accuracy": accuracy_score(y_train, y_pred_train),
    "validation_accuracy": accuracy_score(y_val, y_pred_val),
    "test_accuracy": accuracy_score(y_test, y_pred_test),
    "f1": f1_score(y_test, y_pred_test),
    "kappa": cohen_kappa_score(y_test, y_pred_test),
    "recall": recall_score(y_test, y_pred_test),
    "precision": precision_score(y_test, y_pred_test),
    "auc": roc_auc_score(y_test, y_prob_test),
}

print("Bagging Classifier Results:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


Bagging Classifier Results:
train_accuracy: 0.9996
validation_accuracy: 0.8577
test_accuracy: 0.8925
f1: 0.9227
kappa: 0.7466
recall: 0.9239
precision: 0.9215
auc: 0.9466


# CDIL 

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    recall_score, precision_score, roc_auc_score
)
from sklearn.ensemble import RandomForestClassifier

# -------------------------------------------------------
# Load and Prepare Data
# -------------------------------------------------------
df = pd.read_csv("weather_prediction_dataset.csv")

basel_cols = [c for c in df.columns if c.startswith("BASEL_")]
feature_cols = [c for c in basel_cols if c != "BASEL_cloud_cover"]

X_raw = df[feature_cols].values
df["target"] = (df["BASEL_cloud_cover"] > 4).astype(int)
y_raw = df["target"].values

# -------------------------------------------------------
# Create CDIL-style features without deep learning
# -------------------------------------------------------

def create_cdil_features(data, labels,
                         windows=[5, 10, 20],
                         dilations=[1, 2, 3]):
    """
    Simulates CDIL:
    C = windowed local features
    D = dilated windows
    I = multiple windows (inception)
    L = lag memory from stacking all sequences
    """
    X_features = []
    y_out = []

    max_w = max(windows) * max(dilations)

    for i in range(max_w, len(data)):

        feats = []

        # INCEPTION: multiple window sizes
        for w in windows:

            # DILATION: multiple dilation rates
            for d in dilations:
                idxs = np.arange(i - w * d, i, d)  # dilated sampling
                window_vals = data[idxs].flatten()
                feats.append(window_vals)

        feats = np.concatenate(feats)
        X_features.append(feats)
        y_out.append(labels[i])

    return np.array(X_features), np.array(y_out)


# Generate CDIL features
X, y = create_cdil_features(X_raw, y_raw)

# -------------------------------------------------------
# Train/Val/Test Split
# -------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

# -------------------------------------------------------
# CDIL Classifier (uses RandomForest)
# -------------------------------------------------------
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42
)

clf.fit(X_train, y_train)

# -------------------------------------------------------
# Predictions
# -------------------------------------------------------
y_pred_train = clf.predict(X_train)
y_pred_val = clf.predict(X_val)
y_pred_test = clf.predict(X_test)
y_prob_test = clf.predict_proba(X_test)[:, 1]

# -------------------------------------------------------
# Metrics
# -------------------------------------------------------
metrics = {
    "train_accuracy": accuracy_score(y_train, y_pred_train),
    "validation_accuracy": accuracy_score(y_val, y_pred_val),
    "test_accuracy": accuracy_score(y_test, y_pred_test),
    "f1": f1_score(y_test, y_pred_test),
    "kappa": cohen_kappa_score(y_test, y_pred_test),
    "recall": recall_score(y_test, y_pred_test),
    "precision": precision_score(y_test, y_pred_test),
    "auc": roc_auc_score(y_test, y_prob_test),
}

print("CDIL-Style Model Results:")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


CDIL-Style Model Results:
train_accuracy: 1.0000
validation_accuracy: 0.7124
test_accuracy: 0.7593
f1: 0.8407
kappa: 0.3620
recall: 0.9171
precision: 0.7760
auc: 0.7591


In [3]:
import pandas as pd

df = pd.read_csv("weather_prediction_dataset.csv")

# Filter for BASEL columns
basel_cols = [c for c in df.columns if c.startswith("BASEL_")]
feature_cols = [c for c in basel_cols if c == "BASEL_cloud_cover"]

# Subset dataframe to the feature columns
df_features = df[feature_cols]

stats = pd.DataFrame({
    "min": df_features.min(),
    "max": df_features.max(),
    "mean": df_features.mean(),
    "median": df_features.median(),
    "std_dev": df_features.std()
})

print("\nDescriptive Statistics for BASEL Feature Columns:")
print(stats)



Descriptive Statistics for BASEL Feature Columns:
                   min  max      mean  median   std_dev
BASEL_cloud_cover    0    8  5.418446     6.0  2.325497
